In [73]:
import jieba
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset,DataLoader
import os
from sklearn.model_selection import train_test_split
from torch import nn
import datetime
import time
from torch import nn
from opencc import OpenCC
from tqdm import tqdm
import random
raw_path='.//data//'
config_path='.//dict//'

In [1]:
#实现token
class Tokenizer(object):
    """
    定义分词器
    """

    def __init__(self, data_file,saved_dict=''):
        """
        初始化
        """
        self.data_file = data_file
        self.saved_dict = saved_dict
        # 输入的词表
        self.input_word2idx = {}
        self.input_idx2word = {}
        self.input_dict_len = None
        self.input_embed_dim = 256
        self.input_hidden_size = 256
        # 输出的词表
        self.output_word2idx = {}
        self.output_idx2word = {}
        self.output_dict_len = None
        self.output_embed_dim = 256
        self.output_hidden_size = 256
        # 推理时，输出的最大长度
        self.output_max_len = 100

        # 英文的标点符号
        self.punctuations = [".", ",", "?", "!"]
        # 繁体转简体
        self.opencc = OpenCC("t2s")

    def build_dict(self):
        """
        构建字典
        """
        if os.path.exists(self.saved_dict):
            self.load()
            print("加载本地字典成功")
            return

        input_words = {"<UNK>", "<PAD>"}
        output_words = {"<UNK>", "<PAD>", "<SOS>", "<EOS>"}

        with open(file=self.data_file, mode="r", encoding="utf8") as f:
            for line in tqdm(f.readlines()):
                if line:
                    input_sentence, output_sentence = line.strip().split("\t")
                    input_sentence_words = self.split_input(input_sentence)
                    output_sentence_words = self.split_output(output_sentence)
                    input_words = input_words.union(set(input_sentence_words))
                    output_words = output_words.union(set(output_sentence_words))
        # 输入字典
        self.input_word2idx = {word: idx for idx, word in enumerate(input_words)}
        self.input_idx2word = {idx: word for word, idx in self.input_word2idx.items()}
        self.input_dict_len = len(self.input_word2idx)

        # 输出字典
        self.output_word2idx = {word: idx for idx, word in enumerate(output_words)}
        self.output_idx2word = {idx: word for word, idx in self.output_word2idx.items()}
        self.output_dict_len = len(self.output_word2idx)

        # 保存
        self.save()
        print("保存字典成功")
        
    def split_input(self, sentence):
        """
        预处理
            输入：I'm a student.
            输出：['i', 'm', 'a', 'student', '.']
        """
        # 英文变小写
        sentence = sentence.lower()
        # 把缩写拆开为两个词
        sentence = sentence.replace("'", " ")
        # 把标点符号和单词分开
        sentence = "".join(
            [
                " " + char + " " if char in self.punctuations else char
                for char in sentence
            ]
        )
        # 切分单词
        words = [word for word in sentence.split(" ") if word]
        # 返回结果（列表形式）
        return words

    def split_output(self, sentence):
        """
        切分汉语
            输入：我爱北京天安门
            输出：['我', '爱', '北京', '天安门']
        """
        # 繁体转简体
        sentence = self.opencc.convert(sentence)
        # jieba 分词
        words = jieba.lcut(sentence)
        # 返回结果（列表形式）
        return words
    
    def encode_input(self, input_sentence, input_sentence_len):
        """
        将输入的句子，转变为指定长度的序列号
        输入：["i", "m", "a", "student"]
        输出：[5851, 4431, 6307, 1254, 2965]
        """
        # 变索引号
        input_idx = [
            self.input_word2idx.get(word, self.input_word2idx.get("<UNK>"))
            for word in input_sentence
        ]
        # 填充PAD
        input_idx = (
                            input_idx + [self.input_word2idx.get("<PAD>")] * input_sentence_len
                    )[:input_sentence_len]

        return input_idx

    def encode_output(self, output_sentence, output_sentence_len):
        """
        将输出的句子，转变为指定长度的序列号
        输入：["我", "爱", "北京", "天安门"]
        输出：[11642, 10092, 5558, 3715, 10552, 1917]
        """
        # 添加结束和开始标识符 <EOS>
        output_sentence = output_sentence + ["<EOS>"]
        output_sentence_len += 1
        # 变 索引号
        output_idx = [
            self.output_word2idx.get(word, self.output_word2idx.get("<UNK>"))
            for word in output_sentence
        ]
        # 填充 PAD
        output_idx = (
                             output_idx + [self.output_word2idx.get("<PAD>")] * output_sentence_len
                     )[:output_sentence_len]
        return output_idx

    def decode_output(self, pred):
        """
        把预测结果转换为输出文本
        输入：[6360, 7925, 8187, 7618, 1653, 4509]
        输出：['我', '爱', '北京', '<UNK>']
        """
        raw_result = []
        for idx in pred:
            raw_result.append(self.output_idx2word.get(idx))

        final_result = []
        for idx in pred:
            if idx == self.output_word2idx.get("<EOS>"):
                break
            elif idx == self.output_word2idx.get("<PAD>"):
                continue
            final_result.append(self.output_idx2word.get(idx))

        return raw_result, final_result
    
    def save(self):
        """
        保存字典
        """
        state_dict = {
            "input_word2idx": self.input_word2idx,
            "input_idx2word": self.input_idx2word,
            "input_dict_len": self.input_dict_len,
            "output_word2idx": self.output_word2idx,
            "output_idx2word": self.output_idx2word,
            "output_dict_len": self.output_dict_len,
        }
        # 保存到文件
        torch.save(obj=state_dict, f=self.saved_dict)

    def load(self):
        """
        加载字典
        """
        if os.path.exists(self.saved_dict):
            state_dict = torch.load(f=self.saved_dict)
            self.input_word2idx = state_dict.get("input_word2idx")
            self.input_idx2word = state_dict.get("input_idx2word")
            self.input_dict_len = state_dict.get("input_dict_len")
            self.output_word2idx = state_dict.get("output_word2idx")
            self.output_idx2word = state_dict.get("output_idx2word")
            self.output_dict_len = state_dict.get("output_dict_len")
            
def get_tokenizer(data_file, saved_dict):
    """
    获取分词器
    """
    # 定义分词器
    tokenizer = Tokenizer(data_file=data_file, saved_dict=saved_dict)
    tokenizer.build_dict()
    return tokenizer

In [4]:
class Encoder(nn.Module):
    """
        定义一个 编码器
    """

    def __init__(self, tokenizer):
        super(Encoder, self).__init__()
        self.tokenizer = tokenizer
        # 嵌入层
        self.embed = nn.Embedding(num_embeddings=self.tokenizer.input_dict_len,
                                  embedding_dim=self.tokenizer.input_embed_dim,
                                  padding_idx=self.tokenizer.input_word2idx.get("<PAD>"))
        # GRU单元
        self.gru = nn.GRU(input_size=self.tokenizer.input_embed_dim,
                          hidden_size=self.tokenizer.input_hidden_size,
                          batch_first=False)

    def forward(self, x, x_len):
        # [seq_len, batch_size] --> [seq_len, batch_size, embed_dim]
        x = self.embed(x)
        # 压紧被填充的序列
        x = nn.utils.rnn.pack_padded_sequence(input=x,
                                              lengths=x_len,
                                              batch_first=False)
        out, hn = self.gru(x)
        # 填充被压紧的序列
        out, out_len = nn.utils.rnn.pad_packed_sequence(sequence=out,
                                                        batch_first=False,
                                                        padding_value=self.tokenizer.input_word2idx.get("<PAD>"))
        # out: [seq_len, batch_size, hidden_size]
        # hn: [1, batch_size, hidden_size]
        return out, hn

In [70]:
class Decoder(nn.Module):
    def __init__(self, tokenizer):
        super(Decoder, self).__init__()
        self.tokenizer = tokenizer
        # 嵌入
        self.embed = nn.Embedding(
            num_embeddings=self.tokenizer.output_dict_len,
            embedding_dim=self.tokenizer.output_embed_dim,
            padding_idx=self.tokenizer.output_word2idx.get("<PAD>"),
        )
        # 抽取特征
        self.gru = nn.GRU(
            input_size=self.tokenizer.output_embed_dim,
            hidden_size=self.tokenizer.output_hidden_size,
            batch_first=False,
        )
        # 转换维度，做概率输出
        self.fc = nn.Linear(
            in_features=self.tokenizer.output_hidden_size,
            out_features=self.tokenizer.output_dict_len,
        )

    def forward_step(self, decoder_input, decoder_hidden):
        """
        单步解码:
            decoder_input: [1, batch_size]
            decoder_hidden: [1, batch_size, hidden_size]
        """
        # [1, batch_size] --> [1, batch_size, embedding_dim]
        decoder_input = self.embed(decoder_input)
        # 输入：[1, batch_size, embedding_dim] [1, batch_size, hidden_size]
        # 输出：[1, batch_size, hidden_size] [1, batch_size, hidden_size]
        # 因为只有1步，所以 out 跟 decoder_hidden是一样的
        out, decoder_hidden = self.gru(decoder_input, decoder_hidden)
        # [batch_size, hidden_size]
        out = out.squeeze(dim=0)
        # [batch_size, dict_len]
        out = self.fc(out)
        # out: [batch_size, dict_len]
        # decoder_hidden: [1, batch_size, hidden_size]
        return out, decoder_hidden

    def forward(self, encoder_hidden, y, y_len):
        """
        训练时的正向传播
            - encoder_hidden: [1, batch_size, hidden_size]
            - y: [seq_len, batch_size]
            - y_len: [batch_size]
        """
        # 计算输出的最大长度（本批数据的最大长度）
        output_max_len = max(y_len.tolist()) + 1
        # 本批数据的批量大小
        batch_size = encoder_hidden.size(1)
        # 输入信号 SOS  读取第0步，启动信号
        # decoder_input: [1, batch_size]
        # 输入信号 SOS [1, batch_size]
#         FloatTensor = torch.cuda.FloatTensor if encoder_hidden.is_cuda else torch.FloatTensor
#         LongTensor = torch.cuda.LongTensor if encoder_hidden.is_cuda else torch.LongTensor
        decoder_input = torch.LongTensor(
            [[self.tokenizer.output_word2idx.get("<SOS>")] * batch_size]
        ).to(encoder_hidden.device)
        # 收集所有的预测结果
        # decoder_outputs: [seq_len, batch_size, dict_len]
        decoder_outputs = torch.zeros(
            output_max_len, batch_size, self.tokenizer.output_dict_len
        ).to(encoder_hidden.device)
     
        # 隐藏状态 [1, batch_size, hidden_size]
        decoder_hidden = encoder_hidden
        # 手动循环
        for t in range(output_max_len):
            # 输入：decoder_input: [batch_size, dict_len], decoder_hidden: [1, batch_size, hidden_size]
            # 返回值：decoder_output_t: [batch_size, dict_len], decoder_hidden: [1, batch_size, hidden_size]
            decoder_output_t, decoder_hidden = self.forward_step(
                decoder_input, decoder_hidden
            )
            # 填充结果张量 [seq_len, batch_size, dict_len]
            decoder_outputs[t, :, :] = decoder_output_t
            # teacher forcing 教师强迫机制
            use_teacher_forcing = random.random() > 0.5
            # 0.5 概率 实行教师强迫
            if use_teacher_forcing:
                # [1, batch_size] 取标签中的下一个词
                decoder_input = y[t, :].unsqueeze(0)
            else:
                # 取出上一步的推理结果 [1, batch_size]
                decoder_input = decoder_output_t.argmax(dim=-1).unsqueeze(0)
        # decoder_outputs: [seq_len, batch_size, dict_len]
        return decoder_outputs

    def batch_infer(self, encoder_hidden):
        """
        推理时的正向传播
            - encoder_hidden: [1, batch_size, hidden_size]
        """
        # 推理时，设定一个最大的固定长度
        output_max_len = self.tokenizer.output_max_len
        # 获取批量大小
        batch_size = encoder_hidden.size(1)
        # 输入信号 SOS [1, batch_size]
#         FloatTensor = torch.cuda.FloatTensor if encoder_hidden.is_cuda else torch.FloatTensor
#         LongTensor = torch.cuda.LongTensor if encoder_hidden.is_cuda else torch.LongTensor
        decoder_input = torch.LongTensor(
            [[self.tokenizer.output_word2idx.get("<SOS>")] * batch_size]
        ).to(encoder_hidden.device)
        # print(decoder_input)
        results = []
        # 隐藏状态
        # encoder_hidden: [1, batch_size, hidden_size]
        decoder_hidden = encoder_hidden
        with torch.no_grad():
            # 手动循环
            for t in range(output_max_len):
                # decoder_input: [1, batch_size]
                # decoder_hidden: [1, batch_size, hidden_size]
                decoder_output_t, decoder_hidden = self.forward_step(
                    decoder_input, decoder_hidden
                )
                # 取出结果 [1, batch_size]
                decoder_input = decoder_output_t.argmax(dim=-1).unsqueeze(0)
                results.append(decoder_input)
            # [seq_len, batch_size]
            results = torch.cat(tensors=results, dim=0)
        return results


In [26]:
class Seq2Seq(nn.Module):

    def __init__(self, tokenizer):
        super(Seq2Seq, self).__init__()
        self.tokenizer = tokenizer
        self.encoder = Encoder(self.tokenizer)
        self.decoder = Decoder(self.tokenizer)

    def forward(self, x, x_len, y, y_len):
        """
            训练时的正向传播
        """
        out, hn = self.encoder(x, x_len)
        results = self.decoder(hn, y, y_len)
        # [seq_len, batch_size, dict_len]
        return results

    def batch_infer(self, x, x_len):
        """
            批量推理
        """
        out, hn = self.encoder(x, x_len)
        preds = self.decoder.batch_infer(hn)
        results = []
        for s in preds.t():
            results.append(self.tokenizer.decode_output(s.tolist()))
        return results


In [40]:
class Seq2SeqDataset(Dataset):
    """
    自定义数据集
    """

    def __init__(self, data_file, tokenizer,preload_file='', part="train"):
        self.data_file = data_file
        self.tokenizer = tokenizer
        self.part = part
        self.data = None
        self.preload_file=preload_file
        self._load_data()


    def _load_data(self):
        if os.path.exists(self.preload_file):
            self.data = torch.load(f=self.preload_file)[self.part]
            print(f"加载本地数据集成功,part={self.part}")
            return

        data = []
        with open(file=self.data_file, mode="r", encoding="utf-8") as f:
            for line in tqdm(f.readlines()):
                if line:
                    input_sentence, output_sentence = line.strip().split("\t")
                    input_sentence = self.tokenizer.split_input(input_sentence)
                    output_sentence = self.tokenizer.split_output(output_sentence)
                    data.append([input_sentence, output_sentence])

            train_data, test_data = train_test_split(data, test_size=0.2, random_state=0)

        if self.part == "train":
            self.data = train_data
        elif self.part == "test":
            self.data = test_data
        else:
            self.data=None

        # 保存数据
        torch.save(obj={'train':train_data,'test':test_data}, f=self.preload_file)
        print('加载数据完成预处理')

    def __getitem__(self, idx):
        """
        返回一个样本
            - 列表格式
            - 内容 + 实际长度
        """

        input_sentence, output_sentence = self.data[idx]
        return (
            input_sentence,
            len(input_sentence),
            output_sentence,
            len(output_sentence),
        )

    def __len__(self):
        return len(self.data)


def collate_fn(batch, tokenizer):
    # 根据 x 的长度来 倒序排列
    batch = sorted(batch, key=lambda ele: ele[1], reverse=True)
    # 合并整个批量的每一部分
    input_sentences, input_sentence_lens, output_sentences, output_sentence_lens = zip(
        *batch
    )

    # 转索引【按本批量最大长度来填充】
    input_sentence_len = input_sentence_lens[0]
    input_idxes = []
    for input_sentence in input_sentences:
        input_idxes.append(tokenizer.encode_input(input_sentence, input_sentence_len))

    # 转索引【按本批量最大长度来填充】
    output_sentence_len = max(output_sentence_lens)
    output_idxes = []
    for output_sentence in output_sentences:
        output_idxes.append(
            tokenizer.encode_output(output_sentence, output_sentence_len)
        )

    # 转张量 [seq_len, batch_size]
    input_idxes = torch.LongTensor(input_idxes).t()
    output_idxes = torch.LongTensor(output_idxes).t()
    input_sentence_lens = torch.LongTensor(input_sentence_lens)
    output_sentence_lens = torch.LongTensor(output_sentence_lens)

    return input_idxes, input_sentence_lens, output_idxes, output_sentence_lens


# def get_dataloader(tokenizer, data_file, part="train",preload_file=None, batch_size=32):
#     dataset = Seq2SeqDataset(data_file=data_file, tokenizer=tokenizer,preload_file=preload_file, part=part)[part]
#     dataloader = DataLoader(
#         dataset=dataset,
#         batch_size=batch_size,
#         shuffle=True if part == "train" else False,
#         collate_fn=lambda batch: collate_fn(batch, tokenizer),
#     )
#     return dataloader

In [113]:
class Translation(object):
    """
    把任务封装为一个类
    """

    def __init__(self, data_file,preload_file='',model_file='',saved_dict=''):
        self.data_file = data_file
        self.preload_file=preload_file
        # 检测设备
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        # 
        self.model_file=model_file
        self.saved_dict=saved_dict
        self.preload_file=preload_file
        self.tokenizer = get_tokenizer(data_file=data_file,saved_dict=self.saved_dict)
        self.model = self._get_model()
        self.optimizer = torch.optim.Adam(params=self.model.parameters(), lr=1e-3)
        self.loss_fn = nn.CrossEntropyLoss(
            ignore_index=self.tokenizer.output_word2idx.get("<PAD>")
        )
        self.train_data=Seq2SeqDataset(data_file=self.data_file, tokenizer=self.tokenizer,preload_file=preload_file, part="train")
        self.test_data=Seq2SeqDataset(data_file=self.data_file, tokenizer=self.tokenizer,preload_file=preload_file, part="test")

    def _get_model(self):
        """
        获取模型
        """
        # 实例化模型
        model = Seq2Seq(tokenizer=self.tokenizer)
        # 加载权重
        if os.path.exists(self.model_file):
            model.load_state_dict(state_dict=torch.load(self.model_file))
            print("加载本地模型成功")
        model=model.to(self.device)
        return model

    def get_acc(self,dataloader):
        self.model.eval()
        accs=[]
        with torch.no_grad():
            for (x, x_len, y, y_len) in dataloader:
                #正向传播
                x = x.to(device=self.device)
                y = y.to(device=self.device)
                results = self.model.batch_infer(x, x_len)
                loss=self.get_loss(decoder_outputs=results, y=y)
                accs.append(loss)

        avg_acc=round(number=sum(accs)/len(accs),ndigits=6)
        return avg_acc
    
    def get_loss(self, decoder_outputs, y):
        decoder_outputs = decoder_outputs.to(device=self.device)
        # [batch_size]
        y = y.contiguous().view(-1)
        # [batch_size, dim]
        decoder_outputs = decoder_outputs.contiguous().view(
            -1, decoder_outputs.size(-1)
        )
        # 计算损失
        loss = self.loss_fn(decoder_outputs, y)
        return loss    
    
    
    def get_real_output(self, y):
        """
        将预测结果转换为真实结果
        """
        y = y.t().tolist()
        results = []
        for s in y:
            results.append(
                [
                    self.tokenizer.output_idx2word.get(idx)
                    for idx in s
                    if idx
                    not in [
                        self.tokenizer.output_word2idx.get("<EOS>"),
                        self.tokenizer.output_word2idx.get("<PAD>"),
                    ]
                ]
            )
        return results

    def get_real_input(self, x):
        """
        将输入转换为字符
        """
        x = x.t().tolist()
        results = []
        for s in x:
            results.append(
                [
                    self.tokenizer.input_idx2word.get(idx)
                    for idx in s
                    if idx not in [self.tokenizer.input_word2idx.get("<PAD>")]
                ]
            )
        return results
    
    
    def train(self,epochs=50,batch_size=32,cnt_interval=10,loss_thred=0.1):

        is_complete=False
        dataloader_train = DataLoader(
            dataset=self.train_data,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=lambda batch: collate_fn(batch,self.tokenizer)
        )
        
        dataloader_eval = DataLoader(
            dataset=self.test_data,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=lambda batch: collate_fn(batch,self.tokenizer)
        )

        for epoch in range(epochs):
            start_time = time.time()
            self.model.train()
            for batch_i, (x, x_len, y, y_len) in enumerate(dataloader_train):
                x = x.to(device=self.device)
                y = y.to(device=self.device)
                #print(f"x.shape={x.shape},y.shape={y.shape},x_len={x_len},y_len={y_len}")

                results = self.model(x, x_len, y, y_len)
                #print(f"results.shape={results.shape},y.shape={y.shape}")
                loss=self.get_loss(decoder_outputs=results, y=y)
                #print(loss)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                if (batch_i+1)%cnt_interval==0:
                    epoch_batches_left = len(dataloader_train) - (batch_i + 1)
                    time_left = datetime.timedelta(seconds=epoch_batches_left * (time.time() - start_time) / (batch_i + 1))
                    print(f"epoch={epoch+1},剩余batch={epoch_batches_left},epoch剩余时间={time_left}秒,loss={loss}")
                
                if loss.item() < loss_thred:
                    is_complete = True
                    print(f"训练提前完成, 本批次损失为：{loss.item()}")
                    break
            with torch.no_grad():
                for (x, x_len, y, y_len) in dataloader_train:
                    x = x.to(device=self.device)
                    y = y.to(device=self.device)
                    x_true = self.get_real_input(x)
                    y_pred = self.model.batch_infer(x, x_len)
                    y_true = self.get_real_output(y)
                    samples = random.sample(population=range(x.size(1)), k=2)
                    print(f"\t第{epoch+1}轮已经训练完毕")
                    for idx in samples:
                        print("\t真实输入：", x_true[idx])
                        print("\t真实结果：", y_true[idx])
                        print("\t预测结果：", y_pred[idx][1])
                        print(
                            "\t----------------------------------------------------------"
                        )
                    break
            if is_complete:
                break

        
        return 

    def infer(self, x="Am I wrong?"):
        """
        单样本推理
        """
        print("输入：", x)
        # 分词
        x = self.tokenizer.split_input(x)
        print("分词：", x)
        # 编码
        x = self.tokenizer.encode_input(x, len(x))
        print("编码：", x)
        # 张量
        x = torch.tensor(data=[x], dtype=torch.long).t().to(device=self.device)
        print("张量：", x)
        # 长度
        x_len = torch.tensor(data=[len(x)], dtype=torch.long)
        # 评估模式
        self.model.eval()
        # 无梯度环境
        with torch.no_grad():
            y_pred = self.model.batch_infer(x, x_len)
            print("原始输出：", y_pred[0][0])
            print("最终输出：", y_pred[0][1])
            return y_pred[0]

In [114]:
#main函数
translation = Translation(data_file=raw_path+'data.txt',saved_dict=config_path+'Tokenizer.txt',preload_file=raw_path+'preload_data.bin',model_file=config_path+'curr_best_model.txt')

加载本地字典成功
加载本地数据集成功,part=train
加载本地数据集成功,part=test


In [115]:
translation.train(epochs=100,batch_size=128)

epoch=1,剩余batch=175,epoch剩余时间=0:00:07.594988秒,loss=7.475583076477051
epoch=1,剩余batch=165,epoch剩余时间=0:00:07.111492秒,loss=5.594187259674072
epoch=1,剩余batch=155,epoch剩余时间=0:00:06.716658秒,loss=5.525553226470947
epoch=1,剩余batch=145,epoch剩余时间=0:00:06.256743秒,loss=5.454877853393555
epoch=1,剩余batch=135,epoch剩余时间=0:00:05.810394秒,loss=5.472748756408691
epoch=1,剩余batch=125,epoch剩余时间=0:00:05.299995秒,loss=5.223767280578613
epoch=1,剩余batch=115,epoch剩余时间=0:00:04.861209秒,loss=5.142542362213135
epoch=1,剩余batch=105,epoch剩余时间=0:00:04.434933秒,loss=5.036355495452881
epoch=1,剩余batch=95,epoch剩余时间=0:00:04.044885秒,loss=4.999759674072266
epoch=1,剩余batch=85,epoch剩余时间=0:00:03.633747秒,loss=5.1170735359191895
epoch=1,剩余batch=75,epoch剩余时间=0:00:03.214088秒,loss=5.104727268218994
epoch=1,剩余batch=65,epoch剩余时间=0:00:02.784706秒,loss=4.832522392272949
epoch=1,剩余batch=55,epoch剩余时间=0:00:02.375575秒,loss=4.872126579284668
epoch=1,剩余batch=45,epoch剩余时间=0:00:01.948177秒,loss=4.827197551727295
epoch=1,剩余batch=35,epoch剩余时间=0:00:01.51

epoch=6,剩余batch=175,epoch剩余时间=0:00:07.140004秒,loss=3.0817408561706543
epoch=6,剩余batch=165,epoch剩余时间=0:00:06.872244秒,loss=2.6542811393737793
epoch=6,剩余batch=155,epoch剩余时间=0:00:06.654662秒,loss=3.0622310638427734
epoch=6,剩余batch=145,epoch剩余时间=0:00:06.187870秒,loss=3.143450975418091
epoch=6,剩余batch=135,epoch剩余时间=0:00:05.840093秒,loss=2.8697807788848877
epoch=6,剩余batch=125,epoch剩余时间=0:00:05.431244秒,loss=2.8981168270111084
epoch=6,剩余batch=115,epoch剩余时间=0:00:05.059995秒,loss=3.010007619857788
epoch=6,剩余batch=105,epoch剩余时间=0:00:04.584559秒,loss=2.981417179107666
epoch=6,剩余batch=95,epoch剩余时间=0:00:04.139886秒,loss=2.8197314739227295
epoch=6,剩余batch=85,epoch剩余时间=0:00:03.671996秒,loss=2.6690337657928467
epoch=6,剩余batch=75,epoch剩余时间=0:00:03.244770秒,loss=2.8197948932647705
epoch=6,剩余batch=65,epoch剩余时间=0:00:02.793914秒,loss=3.146538734436035
epoch=6,剩余batch=55,epoch剩余时间=0:00:02.361617秒,loss=3.041813373565674
epoch=6,剩余batch=45,epoch剩余时间=0:00:01.932430秒,loss=2.7359657287597656
epoch=6,剩余batch=35,epoch剩余时间=0:

epoch=11,剩余batch=175,epoch剩余时间=0:00:07.699993秒,loss=1.508565068244934
epoch=11,剩余batch=165,epoch剩余时间=0:00:07.177558秒,loss=1.5377259254455566
epoch=11,剩余batch=155,epoch剩余时间=0:00:06.799369秒,loss=1.6867241859436035
epoch=11,剩余batch=145,epoch剩余时间=0:00:06.209649秒,loss=1.5382459163665771
epoch=11,剩余batch=135,epoch剩余时间=0:00:05.832016秒,loss=1.6391842365264893
epoch=11,剩余batch=125,epoch剩余时间=0:00:05.500012秒,loss=1.7411909103393555
epoch=11,剩余batch=115,epoch剩余时间=0:00:05.004152秒,loss=1.8480198383331299
epoch=11,剩余batch=105,epoch剩余时间=0:00:04.599006秒,loss=1.805938720703125
epoch=11,剩余batch=95,epoch剩余时间=0:00:04.159964秒,loss=1.5581525564193726
epoch=11,剩余batch=85,epoch剩余时间=0:00:03.689015秒,loss=1.672317624092102
epoch=11,剩余batch=75,epoch剩余时间=0:00:03.256376秒,loss=1.6615322828292847
epoch=11,剩余batch=65,epoch剩余时间=0:00:02.816134秒,loss=1.8071688413619995
epoch=11,剩余batch=55,epoch剩余时间=0:00:02.378546秒,loss=1.5066888332366943
epoch=11,剩余batch=45,epoch剩余时间=0:00:01.931470秒,loss=1.5683300495147705
epoch=11,剩余batc

epoch=16,剩余batch=175,epoch剩余时间=0:00:07.927497秒,loss=1.054688811302185
epoch=16,剩余batch=165,epoch剩余时间=0:00:07.334244秒,loss=0.9591330289840698
epoch=16,剩余batch=155,epoch剩余时间=0:00:06.928494秒,loss=0.8483436107635498
epoch=16,剩余batch=145,epoch剩余时间=0:00:06.452495秒,loss=0.9218825101852417
epoch=16,剩余batch=135,epoch剩余时间=0:00:05.921094秒,loss=0.9211506247520447
epoch=16,剩余batch=125,epoch剩余时间=0:00:05.470830秒,loss=0.9084557890892029
epoch=16,剩余batch=115,epoch剩余时间=0:00:05.017282秒,loss=1.0638726949691772
epoch=16,剩余batch=105,epoch剩余时间=0:00:04.524184秒,loss=1.0861377716064453
epoch=16,剩余batch=95,epoch剩余时间=0:00:04.069163秒,loss=1.0472986698150635
epoch=16,剩余batch=85,epoch剩余时间=0:00:03.630357秒,loss=1.1035116910934448
epoch=16,剩余batch=75,epoch剩余时间=0:00:03.178642秒,loss=1.02864408493042
epoch=16,剩余batch=65,epoch剩余时间=0:00:02.756004秒,loss=0.9690570831298828
epoch=16,剩余batch=55,epoch剩余时间=0:00:02.339618秒,loss=1.0177838802337646
epoch=16,剩余batch=45,epoch剩余时间=0:00:01.906716秒,loss=1.0380429029464722
epoch=16,剩余batc

epoch=21,剩余batch=175,epoch剩余时间=0:00:08.154993秒,loss=0.6045258045196533
epoch=21,剩余batch=165,epoch剩余时间=0:00:07.350742秒,loss=0.5087184906005859
epoch=21,剩余batch=155,epoch剩余时间=0:00:06.778661秒,loss=0.516248881816864
epoch=21,剩余batch=145,epoch剩余时间=0:00:06.452495秒,loss=0.5464127063751221
epoch=21,剩余batch=135,epoch剩余时间=0:00:05.875851秒,loss=0.6056196689605713
epoch=21,剩余batch=125,epoch剩余时间=0:00:05.477584秒,loss=0.7355743646621704
epoch=21,剩余batch=115,epoch剩余时间=0:00:05.027537秒,loss=0.5150970816612244
epoch=21,剩余batch=105,epoch剩余时间=0:00:04.566502秒,loss=0.5592444539070129
epoch=21,剩余batch=95,epoch剩余时间=0:00:04.081030秒,loss=0.542820155620575
epoch=21,剩余batch=85,epoch剩余时间=0:00:03.639053秒,loss=0.6338152885437012
epoch=21,剩余batch=75,epoch剩余时间=0:00:03.197890秒,loss=0.6716632843017578
epoch=21,剩余batch=65,epoch剩余时间=0:00:02.755587秒,loss=0.5463464260101318
epoch=21,剩余batch=55,epoch剩余时间=0:00:02.324912秒,loss=0.6585516929626465
epoch=21,剩余batch=45,epoch剩余时间=0:00:01.910650秒,loss=0.6108644604682922
epoch=21,剩余bat

epoch=26,剩余batch=175,epoch剩余时间=0:00:07.472509秒,loss=0.2997220456600189
epoch=26,剩余batch=165,epoch剩余时间=0:00:07.140821秒,loss=0.3833243250846863
epoch=26,剩余batch=155,epoch剩余时间=0:00:06.688528秒,loss=0.4422428011894226
epoch=26,剩余batch=145,epoch剩余时间=0:00:06.189880秒,loss=0.4117439389228821
epoch=26,剩余batch=135,epoch剩余时间=0:00:05.779492秒,loss=0.38618364930152893
epoch=26,剩余batch=125,epoch剩余时间=0:00:05.355317秒,loss=0.37640994787216187
epoch=26,剩余batch=115,epoch剩余时间=0:00:05.000121秒,loss=0.43454739451408386
epoch=26,剩余batch=105,epoch剩余时间=0:00:04.538037秒,loss=0.34943386912345886
epoch=26,剩余batch=95,epoch剩余时间=0:00:04.078193秒,loss=0.23159784078598022
epoch=26,剩余batch=85,epoch剩余时间=0:00:03.618919秒,loss=0.3933883309364319
epoch=26,剩余batch=75,epoch剩余时间=0:00:03.209012秒,loss=0.4772418439388275
epoch=26,剩余batch=65,epoch剩余时间=0:00:02.769298秒,loss=0.2987924814224243
epoch=26,剩余batch=55,epoch剩余时间=0:00:02.339848秒,loss=0.439527690410614
epoch=26,剩余batch=45,epoch剩余时间=0:00:01.937427秒,loss=0.4505573511123657
epoch=26

epoch=31,剩余batch=175,epoch剩余时间=0:00:07.192496秒,loss=0.3383615016937256
epoch=31,剩余batch=165,epoch剩余时间=0:00:07.284745秒,loss=0.18761780858039856
epoch=31,剩余batch=155,epoch剩余时间=0:00:06.788996秒,loss=0.2119825929403305
epoch=31,剩余batch=145,epoch剩余时间=0:00:06.263995秒,loss=0.21041175723075867
epoch=31,剩余batch=135,epoch剩余时间=0:00:05.810395秒,loss=0.15945656597614288
epoch=31,剩余batch=125,epoch剩余时间=0:00:05.506246秒,loss=0.18117226660251617
epoch=31,剩余batch=115,epoch剩余时间=0:00:05.033711秒,loss=0.3047710061073303
epoch=31,剩余batch=105,epoch剩余时间=0:00:04.570121秒,loss=0.21676281094551086
epoch=31,剩余batch=95,epoch剩余时间=0:00:04.125108秒,loss=0.2182653546333313
epoch=31,剩余batch=85,epoch剩余时间=0:00:03.674547秒,loss=0.2066984921693802
epoch=31,剩余batch=75,epoch剩余时间=0:00:03.210302秒,loss=0.1985006332397461
epoch=31,剩余batch=65,epoch剩余时间=0:00:02.766532秒,loss=0.17728480696678162
epoch=31,剩余batch=55,epoch剩余时间=0:00:02.331341秒,loss=0.1919226348400116
epoch=31,剩余batch=45,epoch剩余时间=0:00:01.917142秒,loss=0.24629488587379456
epoch